# Weka gateway benchmarking

Generate output JSON that we can then aggregate, plot, and do statistics on.

## Methodology

This test uses the Butler to get handles for a large number of queries from one or more datasets.  Acquiring the query handles does not pull the actual query result data.  We then select a much smaller sample of those queries to retrieve the result data.

For each query within a given dataset, we measure the size of the HTTP object (referenced with a signed URL via the Weka-S3 gateway at USDF) to retrieve, and fetch the object.  The fetch is performed with `curl`, because we believe that `curl` redirected to `/dev/null` will be much faster than pulling an HTTP object with a Python library.

After all objects have been fetched, the retrieval is (optionally) repeated, to give us insight into the cache size and replacement policy.

This process is then repeated for each dataset.

Statistics (mean, median, population variance, population standard deviation, and coefficient of variation) are collected for each of: object size, measured pull speed, reported pull speed, and ratio of reported speed to measured speed.

If `recheck` is enabled, statistics will be generated for the ratio of reported speeds and of measured speeds between a first and second fetch, to determine cache effects.

The results are then persisted into a file in a shared directory, so we can eventually aggregate over those files, allowing us to test performance of many retrievals in parallel via the `mobu` testing framework.

In [ ]:
# Imports
import json
import sys

from pathlib import Path

from weka_helpers import Benchmarker, Dataset, DatasetList

## Set up input parameters

In [ ]:
# Set up report output directory
output_dir=Path("/rubin/bench/weka/current")  # Only on data-int/dev

# Butler setup
butler_config="dp2"
butler_collections="dp2"

# Unset this to *not* repeat the test with the same files
#  (to test whether Weka caching affects speed)
recheck_speeds = True

# Unset this to disable diagnostic output
debug = True

# Define our input data parameters

data_list: DatasetList = [ 
    Dataset(
        display_name = "datasets",
        dataset_name = "visit_image",  # 30 MB each-ish
        count = 100,
        limit = 10_000,
    ),
    Dataset(
        display_name = "big_datasets",
        dataset_name = "run_provenance",  # 1GB each-ish, variable
        count = 10,
        limit = 10_000,
    ),
]

# anything else with a "+" on this page is also hosted at USDF: https://rubinobs.atlassian.net/wiki/x/CgCveg
# (anything without a +/pencil is on GCS)

## Fetch results

In [ ]:
benchmarker=Benchmarker(
    butler_config=butler_config,
    butler_collections=butler_collections,
    datasetlist=data_list,
    output_dir=output_dir,
    debug=debug,
    recheck=recheck_speeds
)
res=await benchmarker.generate_results()

print(json.dumps(convert_results_to_primitive(res),sort_keys=True,indent=2))